<a href="https://colab.research.google.com/github/viswatejareddygudddeti/ml-project/blob/main/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving movie recomendation.csv to movie recomendation.csv


In [ ]:
df = pd.read_csv("movie recomendation.csv")

df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020.0,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021.0,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021.0,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021.0,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021.0,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
df = df[['title','listed_in','cast','director','description']]

df = df.fillna('')

df['content'] = df['listed_in'] + " " + df['cast'] + " " + df['director'] + " " + df['description']

df.head()

,title,listed_in,cast,director,description,content
0,Dick Johnson Is Dead,Documentaries,,Kirsten Johnson,"As her father nears the end of his life, filmm...",Documentaries Kirsten Johnson As her father n...
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysteries","Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",,"After crossing paths at a party, a Cape Town t...","International TV Shows, TV Dramas, TV Mysterie..."
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act...","Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Julien Leclercq,To protect his family from a powerful drug lor...,"Crime TV Shows, International TV Shows, TV Act..."
3,Jailbirds New Orleans,"Docuseries, Reality TV",,,"Feuds, flirtations and toilet talk go down amo...","Docuseries, Reality TV Feuds, flirtations an..."
4,Kota Factory,"International TV Shows, Romantic TV Shows, TV ...","Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",,In a city of coaching centers known to train I...,"International TV Shows, Romantic TV Shows, TV ..."


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z ]','', text)
    return text

df['content'] = df['content'].apply(clean_text)

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['content'])

tfidf_matrix.shape

(8807, 49821)

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

cosine_sim

array([[1.        , 0.        , 0.        , ..., 0.        , 0.01007725,
        0.        ],
       [0.        , 1.        , 0.02281725, ..., 0.        , 0.        ,
        0.00307711],
       [0.        , 0.02281725, 1.        , ..., 0.        , 0.0050369 ,
        0.00658337],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.04555003,
        0.00191173],
       [0.01007725, 0.        , 0.0050369 , ..., 0.04555003, 1.        ,
        0.00745303],
       [0.        , 0.00307711, 0.00658337, ..., 0.00191173, 0.00745303,
        1.        ]])

In [ ]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def recommend_movie(title, cosine_sim=cosine_sim):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]

    movie_indices = [i[0] for i in sim_scores]

    return df['title'].iloc[movie_indices]

In [ ]:
recommend_movie("Blood & Water")

,title
1514,Diamond City
1593,Kings of Jo'Burg
4475,Shirkers
4271,Lion Pride
108,Dive Club


In [ ]:
X = tfidf_matrix
y = df['listed_in']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()

model.fit(X_train, y_train)

RandomForestClassifier()

In [ ]:
params = {
    'n_estimators':[50,100],
    'max_depth':[10,20]
}

grid = GridSearchCV(
    RandomForestClassifier(),
    params,
    cv=3
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

best_model

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


RandomForestClassifier(max_depth=20, n_estimators=50)

In [ ]:
y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.21679909194097616
                                                                                 precision    recall  f1-score   support

                                                                                      0.10      1.00      0.18       134
                                                             Action & Adventure       0.00      0.00      0.00        27
                             Action & Adventure, Anime Features, Classic Movies       0.00      0.00      0.00         1
                       Action & Adventure, Anime Features, International Movies       0.83      0.62      0.71         8
                           Action & Adventure, Anime Features, Sci-Fi & Fantasy       0.00      0.00      0.00         1
                   Action & Adventure, Children & Family Movies, Classic Movies       0.00      0.00      0.00         1
                         Action & Adventure, Children & Family Movies, Comedies       0.00      0.00      0.00         1
 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import pickle

pickle.dump(best_model, open("movie_model.pkl","wb"))
pickle.dump(tfidf, open("tfidf.pkl","wb"))

In [18]:
!streamlit run app.py &>/dev/null&

In [19]:
!pip install streamlit pyngrok

In [20]:
%%writefile app.py

import streamlit as st

st.title("🎬 Movie Recommendation System")

movie = st.text_input("Enter movie name")

if st.button("Recommend"):
    st.write("Recommended Movies:")
    st.write("Movie 1")
    st.write("Movie 2")
    st.write("Movie 3")

Writing app.py


In [21]:
!streamlit run app.py &>/dev/null&

In [23]:
import streamlit as st
import pandas as pd
import pickle

model = pickle.load(open("movie_model.pkl","rb"))
tfidf = pickle.load(open("tfidf.pkl","rb"))

st.title("Movie Recommendation System")

movie_input = st.text_input("Enter movie description")

if st.button("Predict Genre"):

    vec = tfidf.transform([movie_input])

    prediction = model.predict(vec)

    st.write("Predicted Genre:", prediction[0])

2026-03-08 16:36:11.052 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 16:36:11.057 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 16:36:11.060 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 16:36:11.073 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 16:36:11.080 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 16:36:11.086 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 16:36:11.093 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 16:36:11.099 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.87.64.204:8502

